In [1]:
from langchain_community.document_loaders import YoutubeLoader
from langchain_core.documents import Document 
import yt_dlp
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.proxies import WebshareProxyConfig
import concurrent.futures
import re
import json
import jsonlines

/home/nando/anaconda3/envs/ds_cont/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
url_videos = [
    # Introducción
    "https://youtu.be/3NNcdq1CtX8",
    "https://youtu.be/ry-65Y2JYrQ",
    "https://youtu.be/HWkmtTrngDU",
    "https://youtu.be/NXSyDy0Vfks",
    "https://youtu.be/3rL7Ulep4k4",
    "https://youtu.be/rA-jsTqwxa4",
    "https://youtu.be/EP1-Qu69VFU",
    "https://youtu.be/Jf4PiW0OfnY",
    "https://youtu.be/LWFpYNGfBT0",
    "https://youtu.be/QVTQJubVBYw",
    "https://youtu.be/hfUwa1f4Fd8",
    # Ingreso de asientos de manera manual: modulo de asientos
    "https://youtu.be/Cvl7SLSgJ5I",
    "https://youtu.be/eq_KUZABIQA",
    "https://youtu.be/GRqeojtQMgY",
    "https://youtu.be/fiKsIy5OSX0",
    "https://youtu.be/CbknY_xcSrk",
    "https://youtu.be/DwnJ9V0SGGM",
    "https://youtu.be/Tamqin7FH5w",
    "https://youtu.be/JXwPfZW9-0M",
    "https://youtu.be/kXgXcKjvc9Y",
    "https://youtu.be/yPoSX7pOvqA",
    "https://youtu.be/QiuoUKGmnNk",
    "https://youtu.be/76mokbpXRA8",
    "https://youtu.be/Zpiuy8sdGYk",
    # Ingreso de asientos de manera manual: modulo de operaciones
    "https://youtu.be/kA_QUFB92VU",
    "https://youtu.be/PR9KiPYE2zI",
    "https://youtu.be/jjMPtyCE47s",
    "https://youtu.be/D7get_huCfw",
    "https://youtu.be/SgI8eYTabG0",
    "https://youtu.be/XimYKEYiMtg",
    "https://youtu.be/3o3k1KCZTfo",
    "https://youtu.be/1gObOfIQdLo",
    "https://youtu.be/prUamckBzkc",
    "https://youtu.be/WrUD9VgIRN4",
    "https://youtu.be/N_KPF2fzuD8",
    "https://youtu.be/ANOxvWFilwY",
    "https://youtu.be/p0DBB7MLTdU",
    # Ingreso de información de forma masiva
    "https://youtu.be/jZ4R4EfIvMg",
    # Generación de reportes
    "https://youtu.be/p_huHy4xrbU",
    # Escales y tips
    "https://youtu.be/wqziZeKGoqY",
    "https://youtu.be/HzJrfwNaK1g",
    "https://youtu.be/n3-0LmJMTdg",
    # Subsistemas en DSCont
        # Subsistema de Planillas
        "https://youtu.be/rQAujK1BHSs",
        "https://youtu.be/jqZnx5Qb4ws",
        "https://youtu.be/T3kKQG60wY4",
        "https://youtu.be/lY0y3DcXo_o",
        "https://youtu.be/2M0lM0kvDGc",
        "https://youtu.be/hxZs0z6ylxo",
        "https://youtu.be/N6MLBPlwbdM",
        "https://youtu.be/OuzRM7V6pMo",
        "https://youtu.be/YOBCLTd_5rA",
        "https://youtu.be/kEAzitlFsk8",
        "https://youtu.be/ZE9H7oSZsgw",
        "https://youtu.be/mgcSVu0fzjQ",
        "https://youtu.be/9pqhaBhUpAI",
        "https://youtu.be/szg8kS_fH1g",
        "https://youtu.be/gBIhzXndtH4",
        "https://youtu.be/6zFH7j8_Tx0",
        "https://youtu.be/D-2VNYP0nPE",
        "https://youtu.be/E6FX_c6WkY8",
        "https://youtu.be/5l2qBQP2y3Y",
        "https://youtu.be/_JdV9u8wJuY",
        "https://youtu.be/ik0OiMnuc4M",
        "https://youtu.be/Bq_zcqrc94c",
        # Subsistema Activos fijos
        "https://youtu.be/kyxqQx0I5_8",
        "https://youtu.be/IxNwpZ7OBQw",
        "https://youtu.be/gKvpBkVirRE",
        "https://youtu.be/TojiXyNEhKw",
        "https://youtu.be/OyP5hNQZg2o",
        "https://youtu.be/rPER-MDGa78",
        "https://youtu.be/oC9FtqTZ2vE",
        "https://youtu.be/UQ7NcojGUv0",
        "https://youtu.be/9-WfmsnXCXU",
        "https://youtu.be/vGodsK9zc90",
        # Subsistema de inventarios-DSCont
        "https://youtu.be/V4Oy_IrJZqc",
        "https://youtu.be/uCfV5wQf-PM",
        "https://youtu.be/37EYnv0DVaM",
        "https://youtu.be/UkfAaO4k_k4",
        "https://youtu.be/shKc63BG6FI",
        "https://youtu.be/5D7PsNOf2Dk",
        "https://youtu.be/825NV1QfAf4",
        "https://youtu.be/YEhdGilhw1U",
        "https://youtu.be/1Tv1E0TGf0k",
        "https://youtu.be/BxdMV9QTBsY",
        "https://youtu.be/0o4pxAxk0cI",
        "https://youtu.be/4eJ8tMuzNJg",
    # Actualización DSCont Golden 9.0
    "https://youtu.be/5jitcksqVck",
    "https://youtu.be/FpOUnFCkGkg",
    "https://youtu.be/OEyMha-QrkQ",
    "https://youtu.be/BbS9-YDINec",
    "https://youtu.be/ZBHlTab95dY",
    "https://youtu.be/HlMitr61yuA",
    "https://youtu.be/qJbiad-PPuw",
    "https://youtu.be/MHyybBZNm78",
    # Actualización DSCont Golden 10.0
    "https://youtu.be/X_-cREU-GtY",
    "https://youtu.be/iRLzKYtW3d4",
    "https://youtu.be/3SEMF8ZP_c0",
    "https://youtu.be/gnfAGA66Gyo",
    "https://youtu.be/wueYt6VATa0",
    "https://youtu.be/nanQdW8TmkA",
    "https://youtu.be/dyg7MJ0qfB8",
    "https://youtu.be/4u2H8gSNNXY",
    "https://youtu.be/BooPf0mDiv4",
    "https://youtu.be/sWUxHBL-WdE",
    "https://youtu.be/AwZ6FMbGRoM",
    # Actualización DSCont Golden 11.0
    "https://youtu.be/pOosgO4Auqw",
        # Registro de ventas RVIE
        "https://youtu.be/-Xfb1mT7VKw",
        "https://youtu.be/jAQw2iaLT34",
            # Corregir observaciones al comparar RVIE
            "https://youtu.be/waKdG6YeMCE",
            "https://youtu.be/AdgYFRu8Des",
            "https://youtu.be/6F0wz17r1nw",
        "https://youtu.be/_dFI1JRgFO0",
        "https://youtu.be/xoI_XmGA8AU",
        "https://youtu.be/vb_awlcxKuM",
        # Registro de compras -RCE
        "https://youtu.be/2pCxOlmio_M",
        "https://youtu.be/cvP5rDkoyXc",
            # Verificar y corregir observaciones al comparar RCE
            "https://youtu.be/piM0fAtTz8o",
            "https://youtu.be/M3eoB9Rg7xI",
            "https://youtu.be/qDBPCuhERBE",
            "https://youtu.be/aX9SCtlu3B8",
        "https://youtu.be/0wEDaYpPbs8",
    # Actualización DSCont Golden 11.2 SIRE API
    "https://youtu.be/OCYcgJKpR3o",
    "https://youtu.be/qBRrqdvzrNQ",
    "https://youtu.be/hodgXWN3mNg",
    "https://youtu.be/YkIZzFFwZto",
    "https://youtu.be/ACpFCmIzSmc",
    "https://youtu.be/Kavef0eEWwQ",
    "https://youtu.be/O3e5UprNKuU",
    "https://youtu.be/v6pFixfVoWs",
    "https://youtu.be/t-QElMn8ib0",
    "https://youtu.be/LeSf56U0cgI",
    "https://youtu.be/2qgZWN6SBvQ",
    "https://youtu.be/ffORpKvzj1U",
    "https://youtu.be/EPcg96EW8Rg",
    "https://youtu.be/nnGGrSabuCo",
    "https://youtu.be/MXzdLZvIh6A",
    "https://youtu.be/agmBQ1tbA64",
    # Actualización DSCont Golden 11.4
    "https://youtu.be/5cSPvAGZ9sw",
    "https://youtu.be/oBUPydilJHc",
    "https://youtu.be/BxTag00lddQ",
    "https://youtu.be/nCHRtaKx8Tg",
    "https://youtu.be/MZIl2tnQ5LQ",
    "https://youtu.be/mxJrllbN8oI",
    "https://youtu.be/fwiNuwor-mY",
    "https://youtu.be/qFHnlE-FZQo",
    "https://youtu.be/iSEXfJETgLM",
    "https://youtu.be/ETm_GE52UhA",
    "https://youtu.be/eCzERP1PJ7s",
    "https://youtu.be/_o8s2W2TIBA"
]

In [3]:
def video_procesing(video_url, 
                    ydl_opts = {
                                    'quiet': True,
                                    'noplaylist': True,
                                }
                   ):
    pattern = r"https://youtu\.be/(.*)" 
    match = re.search(pattern, video_url)
    
    if match:
        video_id = match.group(1)
    
    # Cargar transcripción del video y link
    ytt_api = YouTubeTranscriptApi(
        proxy_config=WebshareProxyConfig(
            proxy_username="oqyjdeuy",
            proxy_password="9yq8lu0wkyxu",
        )
    )
    trasncript = ytt_api.fetch(video_id, languages = ['es'])
    print(f"FULL TRANS: {trasncript}")
    text_content = ""
    for snippets in trasncript:
        text_content = text_content + " " + snippets.text   
        print(f"TEXT CONTENT: {text_content}")
    document = Document(page_content = text_content[1:], metadata = {"source": video_url})
    print(f"FULL DOCUMENT: {document}")

    # Cargar metadatos del video  
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info_dict = ydl.extract_info(video_url, download=False)
        
        title = info_dict.get('title', '')
        description = info_dict.get('description', '')

    document.page_content = f"{title} \n {description} \n {document.page_content}"
    document.metadata['title'] = title
    document.metadata['description'] = description
    return document
    
video_procesing("https://youtu.be/3NNcdq1CtX8")

RetryError: HTTPSConnectionPool(host='www.youtube.com', port=443): Max retries exceeded with url: /api/timedtext?v=3NNcdq1CtX8&ei=znGwaZ_TGaeQsfIPx_Ca0Q0&caps=asr&opi=112496729&xoaf=5&xowf=1&hl=en&ip=0.0.0.0&ipbits=0&expire=1773196350&sparams=ip,ipbits,expire,v,ei,caps,opi,xoaf&signature=E08010A056D6C4545D28731AE78AE547771DAA55.A02519F68E33A8BCBF3CD30422F8CEE31901B841&key=yt8&kind=asr&lang=es (Caused by ResponseError('too many 429 error responses'))

In [ ]:
import time

MAX_RETRIES = 5

documents = []

for video_url in url_videos:
    
    success = False
    
    for attempt in range(MAX_RETRIES):
        try:
            result = video_procesing(video_url)
            documents.append(result)
            
            success = True
            print(f"Successfully processed: {video_url}")
            break 
            
        except Exception as e:
            print(f"Attempt {attempt + 1} failed for {video_url}: {e}")
            
            time.sleep(1) 
    
    if not success:
        print(f"FAILED to process {video_url} after {MAX_RETRIES} attempts.")

print(f"Finished processing. Total documents collected: {len(documents)}")

In [8]:
documents

[Document(metadata={'source': 'https://youtu.be/3NNcdq1CtX8', 'title': 'DS-CONT GOLDEN VID 1 CREACIÓN DE EMPRESAS', 'description': ''}, page_content='DS-CONT GOLDEN VID 1 CREACIÓN DE EMPRESAS \n  \n bienvenido a conocer el sistema contable visual con la herramienta que les ayudará a procesar y generar la información contable brindando reportes oportunos ya sea para declaraciones mensuales anuales o de carácter gerencial para la oportuna toma de decisiones en el siguiente vídeo desarrollaremos una monografía con diferentes operaciones como podemos visualizar con la finalidad de mostrar las bondades del sistema de gijón el flujo de trabajo y 1000 de esto y con es bajo el siguiente esquema hace una creación de empresa lo primero que debemos realizar es la creación o apertura de contabilidad una de las bondades del sistema es que te permitirá crear de una en contabilidad es sin límites fase 2 configuración del sistema en esta opción realizaremos las configuraciones básicas para el manejo d

In [9]:
file_path = "Doc.jsonl"
print(f"Saving documents to {file_path}...")
with jsonlines.open(file_path, mode='w') as writer:
    for doc in documents:
        # Convert the Document object to a dictionary
        writer.write(doc.dict())
print("Save complete.")

Saving documents to Doc.jsonl...
Save complete.


/tmp/ipykernel_49001/1918774191.py:6: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  writer.write(doc.dict())


In [62]:
len(url_videos)

149

In [34]:
documents

[]

In [59]:
loaded_docs = []
with jsonlines.open(file_path, mode='r') as reader:
    for doc_dict in reader:
        # Convert the dictionary back into a Document object
        loaded_docs.append(Document(**doc_dict))

print(f"Loaded {len(loaded_docs)} documents.")
print(f"First document content: {loaded_docs[0].page_content}")
print(f"First document metadata: {loaded_docs[0].metadata}")

Loaded 149 documents.
First document content: DS-CONT GOLDEN VID 1 CREACIÓN DE EMPRESAS 
  
 bienvenido a conocer el sistema contable visual con la herramienta que les ayudará a procesar y generar la información contable brindando reportes oportunos ya sea para declaraciones mensuales anuales o de carácter gerencial para la oportuna toma de decisiones en el siguiente vídeo desarrollaremos una monografía con diferentes operaciones como podemos visualizar con la finalidad de mostrar las bondades del sistema de gijón el flujo de trabajo y 1000 de esto y con es bajo el siguiente esquema hace una creación de empresa lo primero que debemos realizar es la creación o apertura de contabilidad una de las bondades del sistema es que te permitirá crear de una en contabilidad es sin límites fase 2 configuración del sistema en esta opción realizaremos las configuraciones básicas para el manejo del sistema jalisco plan contable 2020 de acuerdo al rubro de empresa a más responsables que son los asient